# 数据分析主 Notebook

本 Notebook 是项目的唯一主分析入口，也是迭代分析的交互式工作台。日常分析可以围绕某个中间步骤局部调试和重跑；交付前必须从头到尾顺序执行一次，确保最终结果可复现。

备注规则：每个大章节只保留一个标题，章节说明使用正文加粗标签描述目的、上游依赖、输入文件/对象、输出文件/对象、缓存策略和调试提示。

## 1. 分析框架确认

**目的：** 确认本次分析基于 `docs/analysis_framework.md` 中的业务分析框架，后续数据加载、质量检查、分析路径和报告结构都围绕该框架展开。

**上游依赖：** 用户已确认的业务背景、决策目标、核心业务问题、业务拆解方式和分析边界。

**输入文件/对象：** `docs/analysis_framework.md`；如尚未创建，则参考 `docs/analysis_framework_template.md`。

**输出文件/对象：** Notebook 中的分析框架确认状态；如业务问题发生变化，需要更新 `docs/analysis_framework.md`。

**是否可缓存：** 不缓存。只要业务问题、分析边界或核心假设变化，就应重新确认本节，并检查后续 Notebook、SQL、Python 和报告是否需要同步修改。

**调试提示：** 如果在后续分析中发现原业务拆解不合适，先回到本节更新框架，再继续调试数据或图表，避免分析结果和业务问题脱节。

## 2. 环境、依赖与配置加载

**目的：** 加载项目路径、依赖、配置和随机种子，为后续局部调试提供稳定的运行上下文。

**上游依赖：** 已确认分析框架；当前 Notebook 运行目录位于项目根目录或 `notebooks/` 目录。

**输入文件/对象：** `configs/analysis_config.yaml`；如后续需要连接数据库，还会依赖本地未提交的 `configs/database.yaml`。

**输出文件/对象：** `PROJECT_ROOT`、`analysis_config`、`RANDOM_SEED`、Python import 路径和基础随机种子状态；本节默认不生成输出文件。

**是否可缓存：** 不建议跨会话缓存。每次重启 Kernel、切换配置或变更项目路径后，都应先运行本节，再局部运行后续章节。

**调试提示：** 如果后续 cell 出现路径、配置、模块导入或随机结果不一致的问题，优先重新运行本节。

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FRAMEWORK_PATH = PROJECT_ROOT / "docs" / "analysis_framework.md"
if not FRAMEWORK_PATH.exists():
    raise FileNotFoundError(
        "请先基于 docs/analysis_framework_template.md 创建并确认 docs/analysis_framework.md"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "analysis_config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    analysis_config = yaml.safe_load(file)

RANDOM_SEED = analysis_config["project"].get("random_seed", 42)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT

## 3. 数据库连接与 SQL 执行说明

**目的：** 通过固定的数据源 profile 读取配置、创建连接，并用项目 SQL 文件执行只读查询。

**上游依赖：** 第 2 节已完成配置加载；数据库连接信息、SQL 文件和查询参数已经准备好。

**输入文件/对象：** 本地 `configs/database.yaml`、`configs/analysis_config.yaml` 中的 `analysis.database_profile`、`sql/` 目录下的 SQL 文件、`src/db/connection.py` 和 `src/db/sql_runner.py`。

**输出文件/对象：** `engine` 数据库连接对象、SQL 查询结果 DataFrame、SQL 执行元数据；本节默认不落地文件，是否保存查询结果由第 5 节或项目代码决定。

**是否可缓存：** 可缓存稳定的 SQL 查询结果到 `data/interim/`，用于后续局部调试；当 SQL 文件、查询参数、数据源 profile、时间范围或源表更新时，缓存必须失效。

**调试提示：** 示例代码默认注释，真实项目中按需启用。涉及写库、删表、更新、建表或覆盖表时，必须先显式确认风险，不应在交互式调试中误执行。

In [ ]:
from src.db.connection import create_engine_from_profile, test_connection
from src.db.sql_runner import run_query_with_metadata

DATABASE_CONFIG_PATH = PROJECT_ROOT / "configs" / "database.yaml"
DATABASE_PROFILE = analysis_config["analysis"].get("database_profile")

# 首次运行前，请先复制 configs/database.example.yaml 为 configs/database.yaml，并选择或填写数据源 profile。
# engine = create_engine_from_profile(DATABASE_CONFIG_PATH, profile=DATABASE_PROFILE)
# test_connection(engine)

In [ ]:
# 示例：执行 SQL 文件并查看元数据。
# example_df, example_meta = run_query_with_metadata(
#     engine=engine,
#     sql_file="sql/extract/01_example_query.sql",
# )
# display(example_meta)
# display(example_df.head())

## 4. 数据源与字段理解

**目的：** 说明数据表、字段、粒度、时间范围和核心口径，确保后续清洗和指标计算建立在正确的数据理解上。

**上游依赖：** 分析框架已确认；数据库查询结果、本地数据文件或数据字典已经可访问。

**输入文件/对象：** 数据库表结构、SQL 查询元数据、`data/raw/`、`data/external/`、业务数据字典或字段说明。

**输出文件/对象：** Notebook 中的数据源说明和字段理解记录；建议按需导出 `reports/tables/data_dictionary.csv`。

**是否可缓存：** 可缓存字段字典、表结构快照和数据画像；当源表 schema、字段含义、统计粒度或时间范围变化时，需要重新检查。

**调试提示：** 不要在字段含义、主键、时间字段和核心口径未确认前推进关键指标分析。字段理解变更通常会影响第 5 节之后的所有步骤。

## 5. 数据加载

**目的：** 加载 SQL 查询结果或本地数据文件，并形成后续质量检查、清洗和分析使用的原始 DataFrame。

**上游依赖：** 数据源与字段理解已完成；数据库连接、SQL 结果或本地文件路径已经确认。

**输入文件/对象：** SQL 查询结果、`data/raw/`、`data/external/`、`configs/analysis_config.yaml` 中的数据时间范围和路径配置。

**输出文件/对象：** 原始 DataFrame 或中间 DataFrame；可按项目需要保存到 `data/interim/`，例如原始抽取快照或标准化后的加载结果。

**是否可缓存：** 可缓存。数据库抽取结果或本地大文件加载结果可以保存到 `data/interim/` 便于局部调试；当源数据、SQL、时间范围、筛选条件或字段选择变化时必须重新加载。

**调试提示：** 迭代分析时可以优先从缓存加载，避免频繁访问数据库；但交付冻结前应确认缓存对应的数据版本和参数仍然有效。

## 6. 数据质量检查

**目的：** 检查缺失、重复、异常、主键、时间范围、枚举值和关键指标汇总，判断数据是否足以支撑后续分析。

**上游依赖：** 第 5 节已完成数据加载，并明确主键、时间字段、粒度和关键字段。

**输入文件/对象：** 已加载的原始 DataFrame 或 SQL 查询结果、字段口径、预期主键/唯一键、枚举值和基础校验规则。

**输出文件/对象：** 数据质量检查结果 DataFrame；建议导出 `reports/tables/data_quality_summary.csv`。

**是否可缓存：** 可缓存，但只对同一份输入数据和同一套校验规则有效。数据加载结果、字段口径或校验规则变化后，应重新运行本节。

**调试提示：** 质量问题可能要求回到第 4 节重新确认字段，也可能要求回到第 5 节调整加载逻辑。任何删除、过滤、填补、去重都必须记录规则和影响行数。

## 7. 数据清洗与转换

**目的：** 执行可复现的数据清洗、字段转换、表关联和分析宽表构建，为 EDA、业务分析、统计分析和建模提供稳定输入。

**上游依赖：** 数据质量检查已完成；清洗规则、异常处理方式、缺失值处理方式和指标口径已经明确。

**输入文件/对象：** 原始或中间 DataFrame、`reports/tables/data_quality_summary.csv`、清洗规则、`src/cleaning/` 和 `src/features/` 中的复用函数。

**输出文件/对象：** 清洗后的 DataFrame、特征或分析宽表；建议保存到 `data/interim/` 或 `data/processed/`。

**是否可缓存：** 强烈建议缓存。清洗后数据和分析宽表是最重要的调试断点；当原始数据、清洗规则、join 逻辑、字段转换或指标口径变化时必须重新生成。

**调试提示：** 这是分析过程中最常回头修改的章节。修改本节后，通常需要重新运行第 8 节到第 14 节的下游分析、导出和报告生成。

## 8. 探索性数据分析

**目的：** 理解分布、趋势、差异、异常点和潜在业务假设，为正式业务问题分析提供方向。

**上游依赖：** 清洗后数据或分析宽表已经生成，并通过基本质量检查。

**输入文件/对象：** `data/processed/` 中的分析数据、清洗后的 DataFrame、业务问题、核心分组维度和时间范围配置。

**输出文件/对象：** 探索性图表、临时汇总表和中间发现；经确认有复用价值的结果可保存到 `reports/figures/` 或 `reports/tables/`。

**是否可缓存：** 可缓存已确认的探索性图表和中间汇总表；当清洗后数据、分组维度、时间窗口或异常值处理逻辑变化时应重新运行。

**调试提示：** 探索阶段允许快速试错，但应标记哪些发现进入正式业务分析，避免把临时图表直接混入最终交付。

## 9. 业务问题分析

**目的：** 围绕 `docs/analysis_framework.md` 中确认的核心业务问题逐项完成分析，并形成可追溯的结果、证据和业务解释。

**上游依赖：** 分析框架、指标口径、清洗后数据和必要的探索性发现已经确认。

**输入文件/对象：** 清洗后的分析数据、`docs/analysis_framework.md`、核心指标口径、业务分组维度和必要的 EDA 中间结果。

**输出文件/对象：** 每个业务问题对应的结果表、图表、关键结果和解释；建议输出到 `reports/tables/`、`reports/figures/`，并沉淀为 `reports/final/report_inputs.yaml` 的素材。

**是否可缓存：** 可按业务问题缓存。当对应问题的输入数据、指标口径、分组维度或业务假设未变化时，可以局部复用；否则需要重新运行该问题及其下游报告素材。

**调试提示：** 如果结果显示原分析路径不合理，应回到第 1 节更新业务分析框架，并同步检查最终报告结构。

## 10. 统计分析，如适用

**目的：** 在业务问题需要验证假设或量化不确定性时，使用合适的统计方法补充分析证据。

**上游依赖：** 业务问题、统计假设、样本定义、指标口径和方法选择依据已经明确。

**输入文件/对象：** 分析数据、业务假设、统计方法参数、分组样本和必要的前提检查结果。

**输出文件/对象：** 统计检验结果、效应量、置信区间、模型摘要或诊断结果；建议导出 `reports/tables/statistical_test_results.csv`，相关图表保存到 `reports/figures/`。

**是否可缓存：** 可缓存同一样本、同一方法和同一参数下的统计结果；当样本筛选、时间窗口、指标口径、检验方法或显著性水平变化时需要重新运行。

**调试提示：** 如果本项目不需要统计分析，应在本节正文说明不适用原因。统计显著性和业务显著性必须分开说明。

## 11. 机器学习建模，如适用

**目的：** 在确有预测、分类、聚类、排序或异常检测需求时，完成建模、评估和解释。

**上游依赖：** 建模业务场景、标签定义、特征窗口、样本切分方式、评估指标和误判成本已经确认。

**输入文件/对象：** 建模样本、标签、特征表、随机种子、训练/验证/测试切分配置和 `src/modeling/` 中的复用函数。

**输出文件/对象：** 模型评估结果、特征解释、适用边界和必要的模型中间产物；建议导出 `reports/tables/model_performance_summary.csv` 和相关评估图表。

**是否可缓存：** 可缓存特征矩阵、数据切分和模型评估结果；当标签定义、特征工程、样本窗口、切分方式、模型参数或随机种子变化时必须重新运行。

**调试提示：** 如果本项目不需要机器学习，应在本节正文说明不适用原因。建模调试时重点检查数据泄漏、样本穿越和特征在预测时点是否可获得。

## 12. 结果验证与稳健性检查

**目的：** 检查核心结论是否稳定，是否受异常值、时间窗口、样本选择、指标口径或模型方法影响。

**上游依赖：** 业务问题分析、统计分析或建模结果已经形成，并明确哪些结论进入最终交付。

**输入文件/对象：** 核心结论、关键指标、分析结果表、统计或模型结果、替代时间窗口、替代样本和替代口径配置。

**输出文件/对象：** 稳健性检查表、敏感性分析图表、结论稳定性说明；建议保存到 `reports/tables/` 和 `reports/figures/`。

**是否可缓存：** 可缓存单项检查结果，但上游数据、口径或核心结论变化后必须重新运行相关检查。

**调试提示：** 如果结论对某个设定高度敏感，应回到对应上游章节检查逻辑，并在最终报告中明确说明局限性。

## 13. 图表、表格和报告输入素材导出

**目的：** 将关键图表、结果表和 `reports/final/report_inputs.yaml` 保存到标准目录，形成最终报告的结构化输入。

**上游依赖：** 业务分析、可选统计/建模、稳健性检查和最终报告结构已经确认。

**输入文件/对象：** 关键分析结果、图表对象、结果 DataFrame、`reports/final/final_report_structure.md` 和 `docs/report_inputs_template.yaml`。

**输出文件/对象：** `reports/figures/` 下的关键图表、`reports/tables/` 下的结果表、`reports/final/report_inputs.yaml`。

**是否可缓存：** 不作为分析缓存。每次准备交付候选版本时都应重新导出，确保图表、表格和 `report_inputs.yaml` 来自同一轮有效分析。

**调试提示：** 报告生成前应检查输出路径是否存在、文件是否最新、是否仍有“待填写”占位内容，以及是否包含敏感明细数据。

## 14. 最终报告脚本生成

**目的：** 运行 `scripts/generate_final_report.py`，基于结构化输入生成结果呈现型最终报告。

**上游依赖：** 图表、表格、`reports/final/report_inputs.yaml` 和 `reports/final/final_report_structure.md` 已经准备完成。

**输入文件/对象：** `reports/final/report_inputs.yaml`、`reports/final/final_report_structure.md`、`scripts/generate_final_report.py`。

**输出文件/对象：** `reports/final/final_analysis_report.md`。

**是否可缓存：** 不缓存。只要报告输入素材、报告结构、图表、表格或洞察内容变化，就应重新运行报告生成脚本。

**调试提示：** 默认报告只呈现结果、图表、表格、口径和输出文件路径，不自动生成洞察或业务解读。如需渲染已准备好的洞察内容，必须由用户明确要求，并显式运行 `python scripts/generate_final_report.py --with-insights`。

## 15. 项目完成检查清单

**目的：** 在交付前确认分析框架、数据处理、结果导出、最终报告和敏感信息检查均已完成。

**上游依赖：** 第 1 节到第 14 节已完成；交付候选版本已经从头顺序执行过一次。

**输入文件/对象：** `docs/analysis_framework.md`、`reports/final/final_report_structure.md`、`notebooks/main_analysis.ipynb`、`reports/figures/`、`reports/tables/`、`reports/final/report_inputs.yaml`、`reports/final/final_analysis_report.md`。

**输出文件/对象：** Notebook 中的交付前检查记录；如项目需要，可另存审计记录到 `logs/`。

**是否可缓存：** 不适用。每次交付前都应重新检查。

**调试提示：** 如果任一检查项未完成，应回到对应章节修正，并重新生成受影响的输出文件。

- [ ] 业务分析框架已与用户确认。
- [ ] `docs/analysis_framework.md` 已创建并保存。
- [ ] 最终报告结构已与用户确认。
- [ ] `reports/final/final_report_structure.md` 已创建并保存。
- [ ] 新增或变更的分析需求已同步到业务分析框架。
- [ ] 分析目标已明确。
- [ ] 数据源、时间范围和数据粒度已说明。
- [ ] 核心指标口径已说明。
- [ ] SQL 文件已保存到 `sql/` 并由 Notebook 调用。
- [ ] 可直接执行脚本已记录在 `docs/script_catalog.md`。
- [ ] 新增、删除、重命名或修改脚本后，`docs/script_catalog.md` 已同步更新。
- [ ] 数据质量检查已完成。
- [ ] 清洗规则和影响行数已记录。
- [ ] 关键图表已导出到 `reports/figures/`。
- [ ] 关键结果表已导出到 `reports/tables/`。
- [ ] `reports/final/report_inputs.yaml` 已生成。
- [ ] 已运行 `scripts/generate_final_report.py`。
- [ ] 最终报告已保存到 `reports/final/final_analysis_report.md`。
- [ ] 默认未生成大模型洞察，除非用户明确要求。
- [ ] 如包含洞察或建议，已确认用户明确要求并说明依据。
- [ ] 未泄露数据库凭据或敏感明细数据。